In [3]:
from mock_generator import MockGenerator
from numcosmo_py import Nc, Ncm, sky_match
import numpy as np

Ncm.cfg_init()
%load_ext autoreload
%autoreload 2    

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
Omega_b = 0.0486
Omega_c = 0.2614
Omega_k = 0.0
H0 = 67.7

cosmo = Nc.HICosmoDEXcdm.new()
cosmo.omega_x2omega_k()
cosmo["Omegab"] = Omega_b
cosmo["Omegac"] = Omega_c
cosmo["Omegak"] = Omega_k
cosmo["H0"] = H0
cosmo["w"] = -1.0

prim = Nc.HIPrimPowerLaw.new()
prim.param_set_by_name("ln10e10ASA", 3.0)
prim.props.n_SA =  0.963 

reion = Nc.HIReionCamb.new()

cosmo.add_submodel(prim)
cosmo.add_submodel(reion)

tf = Nc.TransferFuncEH()

psml = Nc.PowspecMLTransfer.new(tf)
psml.require_kmin(1.0e-6)
psml.require_kmax(1.0e3)


psf = Ncm.PowspecFilter.new(psml, Ncm.PowspecFilterType.TOPHAT)
psf.set_best_lnr0()
psf.prepare(cosmo)

old_amplitude = np.exp(prim.props.ln10e10ASA)
ln10e10ASA = np.log((0.8/ cosmo.sigma8(psf)) ** 2 * old_amplitude)
prim.param_set_desc("ln10e10ASA", {"lower-bound": 2.0,"upper-bound": 5.0,"scale": 1.0,"abstol": 1.0e-50,"fit": True,"value": float(ln10e10ASA)})

dist = Nc.Distance.new(2.0)
dist.prepare(cosmo)

mulf = Nc.MultiplicityFuncDespali.new()
mulf.set_mdef(Nc.MultiplicityFuncMassDef.VIRIAL)
hmf = Nc.HaloMassFunction.new(dist, psf, mulf)

In [5]:
class Completeness_model:
    
    """Class to generate completeness model for halo/cluster mocks."""
    def __init__(self, a):
        self.a = a

    
    def complete(self, mass, z):
        return np.ones_like(mass)

    def incomplete(self, mass, z):
        return  np.full_like(mass, self.a, dtype=float)
    
comp= Completeness_model(0.6)
mock_test = MockGenerator(cosmo=cosmo)
print(mock_test.sky_area())
halos = mock_test.generate_halos_from_hmf(hmf,comp.incomplete)
print(len(halos['z']))
print(len(halos[halos['is_detected'] == 1]['z']))
print(len(halos[halos['is_detected'] == 0]['z']))

401.93756131445895
46307
27853
18454


In [6]:
class Purity_model:
    
    """Class to generate completeness model for halo/cluster mocks."""
    def __init__(self, b):
        self.b = b

    
    def pure(self, mass, z):
        return np.ones_like(mass)

    def impure(self, mass, z):
        return  np.full_like(mass, self.b, dtype=float)

pur= Purity_model(0.8)
clusters = mock_test.generate_clusters_from_halos(halos,2.0,hmf,pur.impure)
clusters

TypeError: bad operand type for unary -: 'HaloMassFunction'

In [5]:
print(pur.impure(14.0, 0.3))
print(pur.impure(14.5, 0.3))

0.5
0.5
